# Reproducing the main results

This notebook checks that the code of this repository produces the numbers in the README. It does
**not** retrain anything: it downloads the LoRA adapters we published on Hugging Face, evaluates
them again on the test set from scratch (no cached results are used), and compares the new
predictions with the ones we saved during our runs.

- **No Hugging Face token is needed.** All datasets, adapters and result tables of the
  [bdatm-project](https://huggingface.co/bdatm-project) organization are public, and this notebook
  never writes to the Hub.
- **GPU:** an NVIDIA L4 or A100 (on Colab: *Runtime > Change runtime type*). The model runs in
  bfloat16, which the free T4 GPU does not support well.
- **Decoding is greedy** (no sampling), so on the same GPU type the predictions should be
  identical to ours, or differ on a few samples. A different GPU type can change a few more,
  because floating-point results on GPUs depend slightly on the hardware.
- **Two torch versions.** The predictions also depend on the torch version, and our results were
  produced with two: Task 1 with torch 2.14.1 (`Task_1.ipynb` upgrades torch), Tasks 2 and 3 with
  torch 2.11.0 (preinstalled on Colab). A Python process can load only one torch version, so this
  notebook runs Tasks 2 and 3 in the notebook itself, with torch 2.11.0, and Task 1 in a separate
  Python process with torch 2.14.1. Everything runs with *Runtime > Run all*: no restart is needed,
  and the output of Task 1 is shown in its cell while it runs.

The `LEVEL` setting in Section 2 chooses how much to run. The outputs saved in this notebook come
from `LEVEL = "tables"`; to check the code quickly, choose `"main"` before running:

| `LEVEL` | What runs | Time on an L4 |
|---|---|---|
| `"main"` (quick check) | The raster LoRA model of each task, on all 400 test samples | about 35 min |
| `"tables"` (the saved outputs) | Every row of the README tables, including the 512px resolution ablation. Task 2 and Task 3 on all 400 samples; Task 1 on all 400 samples for the raster LoRA model and on the first 100 for the other rows | about 2.5 h |
| `"full"` | Every row of the README tables, including the 512px resolution ablation, on all 400 test samples | about 6.5 h |

These times are for the evaluation. Downloading the three datasets without a token takes about
40 minutes more in total (about 10 minutes each for Task 1 and Task 2, 20 for Task 3, measured on
our connection), and installing torch 2.14.1 for Task 1 a few minutes.

Task 1 is the slow one: a FEN answer is about 40 tokens long and is generated one token at a time.
Using 100 samples is still a fair check, because the comparison is done sample by sample against
our saved predictions for those same samples.

## 1. Setup

On Colab this cell clones the repository and installs the pinned packages of `requirements.txt`
except `torch` and `torchvision`. It keeps Colab's preinstalled torch if it is 2.11.0, the version
of our Task 2 and Task 3 runs, and installs 2.11.0 otherwise. It also removes Colab's `torchao`,
which is too old for `peft`, and `torchaudio`, which is not used and would not match the torch of
the Task 1 environment. To run locally, set `CONFIG["colab"] = False`, install the packages with
`pip install -r requirements.txt` (it pins torch 2.11.0) and start Jupyter from the `notebooks/`
folder.

`run_streaming` runs a command and shows its output in the cell while it runs; it is used to
create the Task 1 environment and to run Task 1.

In [ ]:
import codecs
import importlib.metadata
import subprocess
import sys
from pathlib import Path

CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}
# torch versions of our runs (see TORCH_VERSIONS in src/eval/reproduce.py)
KERNEL_TORCH, KERNEL_TORCHVISION = "2.11.0", "0.26.0"   # Tasks 2 and 3, in this notebook
TASK1_TORCH, TASK1_TORCHVISION = "2.14.1", "0.29.1"     # Task 1, in a separate environment


def run_streaming(cmd, **kwargs):
    """Runs a command and shows its output in the cell while it runs."""
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    # read whatever is available, not whole lines: progress bars redraw a line with "\r"
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    assert proc.wait() == 0, f"Command failed: {' '.join(str(c) for c in cmd)}"


if CONFIG["colab"]:
    repo_root = Path(CONFIG["repo_dir"])
    if not repo_root.exists():
        repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"
        print(f"Cloning {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True,
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
    # torch and torchvision are handled below: install everything else at the pinned versions
    requirements = [line for line in (repo_root / "requirements.txt").read_text().splitlines()
                    if line.strip() and not line.startswith(("#", "torch"))]
    Path("/tmp/requirements_colab.txt").write_text("\n".join(requirements))
    !pip install -q -r /tmp/requirements_colab.txt
    # Colab's preinstalled torchao is too old for peft, which then fails to load the adapters;
    # the task notebooks remove it too. torchaudio is not used and would not match torch 2.14.1.
    !pip uninstall -y -q torchao torchaudio
    # Our Task 2 and Task 3 runs used Colab's torch 2.11.0 (CUDA 13.0 build)
    if importlib.metadata.version("torch").split("+")[0] != KERNEL_TORCH:
        !pip install -q torch=={KERNEL_TORCH} torchvision=={KERNEL_TORCHVISION} --index-url https://download.pytorch.org/whl/cu130
else:
    repo_root = Path(".").resolve().parent

print(f"Repository: {repo_root}")

### Task 1 environment

Task 1 runs in a separate Python environment (a virtualenv in `.venv_task1/` of the repository)
with torch 2.14.1, the version of our Task 1 runs. The environment contains only `torch`,
`torchvision` and their CUDA libraries; every other package (transformers, peft, ...) is taken
from the installation above, through a `.pth` file that adds its folders after the environment's
own, so the two evaluations use the same versions of everything except torch.

In [ ]:
import site

task1_venv = repo_root / ".venv_task1"
task1_python = task1_venv / "bin" / "python"
if not task1_python.exists():
    # --without-pip: pip comes from the main installation, through the .pth file below
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(task1_venv)], check=True)
task1_site = Path(subprocess.run(
    [task1_python, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
    capture_output=True, text=True, check=True,
).stdout.strip())
(task1_site / "main_environment.pth").write_text("\n".join(site.getsitepackages()) + "\n")
run_streaming([task1_python, "-m", "pip", "install", "-q",
               f"torch=={TASK1_TORCH}", f"torchvision=={TASK1_TORCHVISION}"])
run_streaming([task1_python, "-c", "import torch, transformers; "
               "print(f'Task 1 environment: torch {torch.__version__}, transformers {transformers.__version__}')"])

In [ ]:
import json
import os

import pandas as pd
import torch
import transformers
import peft

sys.path.insert(0, str(repo_root / "src"))

from eval.reproduce import IMAGE_SIZE, check_token_reordering, make_processors, reproduce_task

print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}")
assert torch.cuda.is_available(), "This notebook needs a GPU (an L4 or A100)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
if not torch.cuda.is_bf16_supported():
    print("WARNING: this GPU does not support bfloat16 well; results may differ from ours.")

## 2. Settings

`LEVEL` chooses which rows to reproduce (see the table at the top). On Colab it is a drop-down menu
next to the code; elsewhere, edit the value. The outputs saved in this notebook come from
`LEVEL = "tables"`; for a quick check choose `"main"`.

`NUM_WORKERS` sets how many processes download and load the datasets in parallel. It only changes
the loading time, not the results; a split stored as a single file is loaded by one process
whatever its value. The other settings (model, image sizes, scan orders, number of shots) are the
ones used in the task notebooks and are defined in `src/eval/reproduce.py`.

In [ ]:
LEVEL = "tables"  # @param ["main", "tables", "full"]
TASK1_SUBSET = 100      # with LEVEL = "tables": test samples for the Task 1 rows other than raster LoRA
NUM_WORKERS = 8  # @param {type:"integer"}

assert LEVEL in ("main", "tables", "full"), f"Unknown LEVEL: {LEVEL}"

## 3. The evaluation code

The code that evaluates the models is in `src/eval/reproduce.py`, so that this notebook (Tasks 2
and 3) and the Task 1 process run exactly the same code:

- `load_task_dataset` loads a task's dataset from the Hub the same way the task notebooks do.
  Task 3 has two images per sample, so its files are downloaded first and the second frame
  (`image_t1`) is added to each row.
- `experiment_rows` lists the rows of the README tables for a task: the model name used in the
  results table, the adapter on the Hub (`None` for the vanilla model), the scan order used at
  test time and the image size. For the learned orders, the order is read from
  `learned_order.json` in the adapter's repository. The last row is the 512px raster LoRA model of
  the resolution ablation (repository `qwen-taskN-standard-lora`, named
  `Qwen + LoRA Fine-Tuning (from HF)` in the results table).
- `evaluate_row` loads the base model and the adapter, sets the scan order and runs the task's
  evaluation function of `src/eval/evaluation.py`, with the processor of the row's image size.
- `compare_with_published` downloads our saved per-sample predictions for the same model and
  compares them with the new ones, sample by sample.
- `reproduce_task` runs the rows selected by `LEVEL` for one task and returns the comparison. It
  first prints a warning if the installed torch is not the one of our runs of that task.

## 4. Processors and token reordering check

Two processors are prepared: one resizes the boards to 256px (all the main experiments) and one to
512px (the resolution ablation). Each prints the token grid it gives: 8x8 tokens at 256px (one per
square) and 16x16 at 512px (2x2 per square).

The scan orders are applied by patching the model's code at run time
(`src/training/token_reordering.py`). `verify_token_reordering` checks that the patch works with
the installed `transformers` version: with no order set the output equals the unpatched model,
the token embeddings and their positions move together, and setting an order changes the output.
The Task 1 process runs the same check at its start.

In [ ]:
processors = make_processors()
check_token_reordering(processors[IMAGE_SIZE])

## 5. Evaluate and compare

For each task: load the dataset and our published results table, then evaluate every selected row
and compare it with our saved predictions. Every row prints a progress bar; Task 1 rows take
about 20-25 minutes each on all 400 samples, Task 2 and Task 3 rows about 2 minutes. The 512px
rows are slower, because each image gives 4 times as many visual tokens.

In the comparison table:

- **ours, same samples**: the score of our saved predictions on the samples evaluated here;
- **reproduced**: the score of the new predictions on the same samples;
- **identical predictions**: how many new predictions are exactly the same string as ours;
- **ours, 400 samples (README)**: the number reported in the README.

### Task 1

Task 1 runs in the Task 1 environment (torch 2.14.1), as a separate Python process:
`python -m eval.reproduce --task task1 ...` runs the same `reproduce_task` used below for Tasks 2
and 3, and writes the comparison to `reproduce_task1.json`, which is then shown here. Its output
(versions, token reordering check, progress bars, results of each row) appears below while it
runs. When it ends, the process exits and frees the GPU.

In [ ]:
TASK = "task1"
output_file = Path(f"reproduce_{TASK}.json")
env = {**os.environ, "PYTHONPATH": str(repo_root / "src"), "PYTHONUNBUFFERED": "1"}
run_streaming([task1_python, "-m", "eval.reproduce", "--task", TASK, "--level", LEVEL,
               "--num-workers", NUM_WORKERS, "--task1-subset", TASK1_SUBSET, "--output", output_file], env=env)

comparison_task1 = json.loads(output_file.read_text())
display(pd.DataFrame(comparison_task1))

### Task 2

In [ ]:
comparison_task2 = reproduce_task("task2", LEVEL, processors, NUM_WORKERS, TASK1_SUBSET)
display(pd.DataFrame(comparison_task2))

### Task 3

In [ ]:
comparison_task3 = reproduce_task("task3", LEVEL, processors, NUM_WORKERS, TASK1_SUBSET)
display(pd.DataFrame(comparison_task3))

## 6. Summary

All the reproduced rows together. The scores should match ours and nearly every prediction should
be identical; on a different GPU type a few predictions can change (see the top of the notebook).

In [ ]:
summary = pd.DataFrame(comparison_task1 + comparison_task2 + comparison_task3)
summary["difference (points)"] = (summary["reproduced (%)"] - summary["ours, same samples (%)"]).round(2)
print(f"LEVEL = {LEVEL!r}, GPU: {torch.cuda.get_device_name(0)}")
display(summary)